#Lab: Spark Operations
##**Course 1, Week 2: Spark Fundamentals**
###Objectives
- Use select() to choose and transform columns
- Use filter() to select rows by condition
- Use groupBy() with aggregation functions
- Perform joins between DataFrames
- Understand lazy evaluation vs actions

## Setup: Create Lab Data

In [0]:
from pyspark.sql import functions as F

# Sales data
sales = spark.createDataFrame(
    [
        ("2024-01-01", "S001", "Electronics", "Laptop", 999.99, 2, "West"),
        ("2024-01-01", "S002", "Books", "Python Guide", 49.99, 5, "East"),
        ("2024-01-02", "S003", "Electronics", "Phone", 699.99, 3, "West"),
        ("2024-01-02", "S004", "Clothing", "Jacket", 129.99, 4, "North"),
        ("2024-01-03", "S005", "Books", "Data Science", 59.99, 8, "East"),
        ("2024-01-03", "S006", "Electronics", "Tablet", 449.99, 2, "South"),
        ("2024-01-04", "S007", "Clothing", "Shoes", 89.99, 6, "West"),
        ("2024-01-04", "S008", "Electronics", "Earbuds", 79.99, 10, "North"),
        ("2024-01-05", "S009", "Books", "ML Handbook", 69.99, 3, "South"),
        ("2024-01-05", "S010", "Clothing", "Hat", 29.99, 15, "East"),
    ],
    ["date", "sale_id", "category", "product", "price", "quantity", "region"],
)

# Region lookup
regions = spark.createDataFrame(
    [
        ("West", "Pacific", "Sarah"),
        ("East", "Atlantic", "Mike"),
        ("North", "Central", "Lisa"),
        ("South", "Gulf", "Tom"),
    ],
    ["region", "territory", "manager"],
)

print(f"Sales: {sales.count()} rows")
print(f"Regions: {regions.count()} rows")

## Part 1: Select Operations
###Use select() to create derived columns.

In [0]:
# Select sale_id, category, product, price, quantity
sales_df = sales.select("sale_id", "category", "product", "price", "quantity")
display(sales_df)

# Create a total_revenue column (price * quantity) and a discounted_price column (price * 0.9)
sales_df = sales_df.withColumn("total_revenue", F.col("price") * F.col("quantity"))
sales_df = sales_df.withColumn("discounted_price", F.col("price") * 0.9)
display(sales_df)


## Part 2: Filter Operations
###Use filter() to find specific rows.

In [0]:
# Filter sales where price > 100
filtered_df = sales_df.filter(F.col("price") > 100)
display(sales_df)

# Filter Electronics sales in the West region
filtered_df = filtered_df.filter((F.col("category") == "Electronics") & (F.col("region") == "West"))
display(sales_df)

# Filter sales between Jan 2 and Jan 4 (inclusive)
filtered_df = filtered_df.filter((F.col("date") >= "2024-01-02") & (F.col("date") <= "2024-01-04"))
display(sales_df)

##Part 3: GroupBy & Aggregations
###Compute summary statistics.

In [0]:
# Total revenue by category
# Columns: category, total_revenue (sum of price*quantity), num_sales (count)
# Order by total_revenue descending

sales_df = sales_df.groupBy("category").agg(
    F.sum(F.col("total_revenue")).alias("total_revenue"),
    F.count(F.col("sale_id")).alias("num_sales"),
)

sales_df = sales_df.orderBy(F.desc("total_revenue"))
display(sales_df)


##Part 4: Joins
###Combine sales with region information.

In [0]:
# Register DataFrames as temporary views first
sales.createOrReplaceTempView("sales")
regions.createOrReplaceTempView("regions")

# Inner join sales with regions on the region column
# Show: sale_id, product, price, region, territory, manager
spark.sql("""
SELECT sales.sale_id, sales.product, sales.price, regions.region, regions.territory, regions.manager
FROM sales
INNER JOIN regions
ON sales.region = regions.region
""").display()

# Find total revenue by territory and manager
# (Join first, then groupBy territory and manager)
spark.sql("""
SELECT regions.territory, regions.manager, SUM(sales.price * sales.quantity) as total_revenue
FROM sales
INNER JOIN regions
ON sales.region = regions.region
GROUP BY regions.territory, regions.manager
""").display()

##Part 5: SQL Equivalent
###Write SQL queries.

In [0]:
%sql
-- Write a SQL query that finds the top 3 products by total revenue
-- (revenue = price * quantity)
SELECT product, sum(price * quantity) as total_revenue 
FROM sales 
GROUP BY product
ORDER BY total_revenue DESC LIMIT 3


##Lab Validation and Cleanup

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Sales data loaded
    checks.append(("Sales data loaded", sales.count() == 10))

    # Check 2: Can perform select
    try:
        result = sales.select("sale_id", "product", "price", "quantity")
        checks.append(("Select operation", len(result.columns) == 4))
    except Exception:
        checks.append(("Select operation", False))

    # Check 3: Can perform filter
    try:
        result = sales.filter(F.col("price") > 100)
        checks.append(("Filter operation", result.count() > 0))
    except Exception:
        checks.append(("Filter operation", False))

    # Check 4: Can perform groupBy
    try:
        result = sales.groupBy("category").agg(F.count("*").alias("n"))
        checks.append(("GroupBy operation", result.count() == 3))
    except Exception:
        checks.append(("GroupBy operation", False))

    # Check 5: Can perform join
    try:
        result = sales.join(regions, "region", "inner")
        checks.append(("Join operation", result.count() == 10))
    except Exception:
        checks.append(("Join operation", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()